# 04 · v2: the same chatbot with RAG
- **Retrieve, then answer**: find the few log windows and runbook sections that match the question, give only those to the model
- Logs are cut into 10-minute windows per file (repeated lines counted); runbooks into sections
- Every chunk becomes 384 numbers with MiniLM; search = cosine similarity + exact-word bonus + same-window evidence from other files, most relevant first
- **Code finds the facts**: pandas counts each exception / version / IP in the sources (first time, last time, how many) and adds them as FACTS
- The model must cite sources [S1], [S2] … or say **Not found in the logs**

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'          # models come from the local cache only
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_colwidth', 120)
from logbot.parse import read_logs
from logbot.rag import RagIndex, AskV2, build_messages
df = read_logs()
index = RagIndex.build(df); index.save()
print(len(index.chunks), 'chunks →', index.vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

100 chunks → (100, 384)


## One log chunk and one runbook chunk

In [2]:
print(next(c['text'] for c in index.chunks if 'SQLTransient' in c['text'])[:600])
print('---')
print(next(c['text'] for c in index.chunks if c['kind'] == 'kb')[:400])
from logbot.rag import facts
print('--- FACTS for the 502 question:')
print(facts(index.search('Why did Apache return 502 errors around 11:02 on Oct 14?'), df))

app/application.log 2026-10-13 10:20–2026-10-13 10:30:
WARN ×1: HikariPool: HikariPool-1 - Pool stats (total=10, active=10, idle=0, waiting=37)
ERROR ×15: OrderService: java.sql.SQLTransientConnectionException: HikariPool-1 - Connection is not available, request timed out after 30000ms.
stack trace: at com.zaxxer.hikari.pool.HikariPool.createTimeoutException(HikariPool.java:696) | at com.zaxxer.hikari.pool.HikariPool.getConnection(HikariPool.java:181)
---
# shop-app architecture

- Apache httpd 2.4 on port 443 terminates TLS and proxies to Tomcat with mod_proxy (127.0.0.1:8080, timeout 60 s).
- shop-app: Java 17, Spring Boot 3 on Tomcat 10, heap -Xmx2g, 40 request threads.
- Database: PostgreSQL 15 through HikariCP (maximumPoolSize 10, connectionTimeout 30000 ms).
- Payments: external gateway over HTTPS, read timeout 5 s.
- Logs: Apache access_log 
--- FACTS for the 502 question:
- AH01102: first 2026-10-14 11:02:10, last 2026-10-14 11:03:30, 9 lines in apache/error_log
- AH00957: firs

## Search only (no model yet)

In [3]:
hits = index.search('Why did Apache return 502 errors around 11:02 on Oct 14?')
pd.DataFrame(hits)[['score', 'file', 'start']].assign(first_line=[h['text'].splitlines()[1][:90] if len(h['text'].splitlines()) > 1 else '' for h in hits])

,score,file,start,first_line
0,0.708,apache/access_log,2026-10-14 11:00,"10.0.5.124 - - [14/Oct/2026:11:02:11 +0000] ""GET /api/products HTTP/1.1"" 503 367 ""-"" ""Mozi"
1,0.671,apache/error_log,2026-10-14 11:00,ERROR ×9: proxy_http: (70007)The timeout specified has expired: AH01102: error reading sta
2,0.670,tomcat/catalina.out,2026-10-14 11:00,"ERROR ×1: Exception in thread ""http-nio-8080-exec-17"" java.lang.OutOfMemoryError: Java hea"
3,0.526,apache/access_log,2026-10-14 16:30,"10.0.4.103 - - [14/Oct/2026:16:30:09 +0000] ""POST /api/checkout HTTP/1.1"" 200 1982 ""-"" ""Mo"
4,0.525,app/application.log,2026-10-14 16:30,ERROR ×5: PaymentClient: Payment authorisation failed: java.net.SocketTimeoutException: Re
5,0.416,kb/runbooks/brute-force-login.md,,


## Retrieve, then answer

In [4]:
from logbot.llm import LocalLLM
llm = LocalLLM()
v2 = AskV2(llm, index, df)
for q in ['Why did Apache return 502 errors around 11:02 on Oct 14?', 'What caused the 500 errors on /api/orders on Oct 13?', 'Was there a disk full error?']:
    a, src, s = v2.ask(q)
    print(f'Q: {q}\nA: {a}')
    print('   sources:', ', '.join(f"{x['id']} {x['file']} {x['start']}" for x in src[:4]), f'({s["seconds"]} s)\n')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Q: Why did Apache return 502 errors around 11:02 on Oct 14?
A: Not found in the logs.
   sources: S1 apache/access_log 2026-10-14 11:00, S2 apache/error_log 2026-10-14 11:00, S3 tomcat/catalina.out 2026-10-14 11:00, S4 apache/access_log 2026-10-14 16:30 (0.2 s)



Q: What caused the 500 errors on /api/orders on Oct 13?
A: The 500 error on `/api/orders` occurred due to a transient connection issue with the database. This was resolved by cancelling the query, moving the reports to the read replica, and setting a 60-second statement timeout for report users.
   sources: S1 apache/access_log 2026-10-13 10:40, S2 apache/access_log 2026-10-13 10:20, S3 app/application.log 2026-10-13 10:20, S4 apache/access_log 2026-10-13 10:30 (0.72 s)

Q: Was there a disk full error?
A: Not found in the logs.
   sources: S1 tomcat/catalina.out 2026-10-14 11:00, S2 app/application.log 2026-10-14 11:00, S3 apache/error_log 2026-10-14 11:00, S4 app/application.log 2026-10-14 09:30 (0.14 s)



In [5]:
hits = index.search('Why did Apache return 502 errors around 11:02 on Oct 14?')
assert any('OutOfMemoryError' in h['text'] for h in hits)
print('✓ v2 retrieves the real cause (OutOfMemoryError) and cites its sources')

✓ v2 retrieves the real cause (OutOfMemoryError) and cites its sources
